# 5.14 Gaussian Processes

A kernel is a prior over smoothness: it says which function values should move together.

Gaussian Processes extend Bayesian regression from finite weights to distributions over functions. This notebook conditions a real RBF GP, then studies posterior error and conditioning on increasingly difficult synthetic surfaces.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 514
rng = np.random.default_rng(SEED)


def rbf_kernel(X, Y, length_scale=1.0, variance=1.0):
    X = np.atleast_2d(X).astype(float)
    Y = np.atleast_2d(Y).astype(float)
    x_norm = np.sum(X * X, axis=1)[:, None]
    y_norm = np.sum(Y * Y, axis=1)[None, :]
    sqdist = x_norm + y_norm - 2.0 * X @ Y.T
    return variance * np.exp(-0.5 * sqdist / (length_scale ** 2))


def gp_posterior(X, y, Xstar, kernel, sigma2=0.1, jitter=1e-8):
    K = kernel(X, X)
    Ks = kernel(Xstar, X)
    Kss = kernel(Xstar, Xstar)
    system = K + (sigma2 + jitter) * np.eye(len(X))
    alpha = np.linalg.solve(system, y)
    mean = Ks @ alpha
    v = np.linalg.solve(system, Ks.T)
    cov = Kss - Ks @ v
    var = np.maximum(np.diag(cov), 0.0)
    return mean, var, cov


def build_gp_ladder():
    xs = np.linspace(-3.0, 3.0, 120)[:, None]
    d1 = {
        "name": "D1 two-point GP exact",
        "X": np.array([[-1.0], [1.0]]),
        "y": np.array([-0.8, 0.9]),
        "Xstar": xs,
        "truth": np.sin(xs[:, 0]),
        "length": 1.0,
        "sigma2": 0.1,
    }
    x2 = np.array([[-2.5], [-0.7], [0.6], [2.2]])
    d2 = {
        "name": "D2 four-point smooth function",
        "X": x2,
        "y": np.sin(x2[:, 0]) + 0.05 * np.array([1.0, -1.0, 0.5, -0.5]),
        "Xstar": xs,
        "truth": np.sin(xs[:, 0]),
        "length": 1.0,
        "sigma2": 0.05,
    }
    x3 = np.array([[-2.8], [-1.7], [-0.2], [0.2], [1.7], [2.8]])
    truth3 = np.sin(xs[:, 0]) + 0.7 * np.exp(-((xs[:, 0] - 1.2) ** 2) / 0.08)
    d3 = {
        "name": "D3 bimodal nonstationary sample",
        "X": x3,
        "y": np.sin(x3[:, 0]) + 0.7 * np.exp(-((x3[:, 0] - 1.2) ** 2) / 0.08),
        "Xstar": xs,
        "truth": truth3,
        "length": 0.7,
        "sigma2": 0.05,
    }
    grid = np.array([[a, b] for a in [-1.0, 0.0, 1.0] for b in [-1.0, 0.0, 1.0]])
    star2 = np.array([[a, b] for a in np.linspace(-1.2, 1.2, 20) for b in np.linspace(-1.2, 1.2, 20)])
    truth4 = np.sin(star2[:, 0]) + 0.5 * np.cos(1.5 * star2[:, 1])
    d4 = {
        "name": "D4 correlated 2-D surface",
        "X": grid,
        "y": np.sin(grid[:, 0]) + 0.5 * np.cos(1.5 * grid[:, 1]),
        "Xstar": star2,
        "truth": truth4,
        "length": 0.9,
        "sigma2": 0.03,
    }
    base = np.linspace(0.0, 1.0, 18)
    X5 = np.column_stack([base, base + 1e-3 * rng.normal(size=18), np.sin(base)])
    Xs5 = np.column_stack([np.linspace(0.0, 1.0, 80), np.linspace(0.0, 1.0, 80), np.sin(np.linspace(0.0, 1.0, 80))])
    truth5 = np.sin(4.0 * Xs5[:, 0]) + 0.2 * Xs5[:, 2]
    d5 = {
        "name": "D5 ill-conditioned higher-dim kernel",
        "X": X5,
        "y": np.sin(4.0 * X5[:, 0]) + 0.2 * X5[:, 2],
        "Xstar": Xs5,
        "truth": truth5,
        "length": 3.0,
        "sigma2": 1e-5,
    }
    return [d1, d2, d3, d4, d5]


def predictive_rmse(mean, truth):
    return float(np.sqrt(np.mean((mean - truth) ** 2)))


## The concept, built once: exact GP conditioning

A GP prior uses a kernel covariance matrix:

$$f(X)\sim\mathcal N(0,K),\quad K_{ij}=k(x_i,x_j),\quad \mu_*=K_{*X}(K_{XX}+\sigma^2I)^{-1}y.$$

The RBF kernel turns distance into covariance.

In [ ]:

distance = 2.0
length_scale = 1.0
covariance = math.exp(-0.5 * distance ** 2 / length_scale ** 2)
print("RBF covariance at distance 2", covariance)
assert np.isclose(covariance, 0.135335, atol=1e-6)

X = np.array([[0.0], [2.0]])
y = np.array([1.0, -1.0])
Xstar = np.array([[1.0]])
kernel = lambda A, B: rbf_kernel(A, B, length_scale=1.0, variance=1.0)
mean, var, cov = gp_posterior(X, y, Xstar, kernel, sigma2=0.1)
manual_mean = kernel(Xstar, X) @ np.linalg.solve(kernel(X, X) + 0.1 * np.eye(2), y)
print("posterior mean", mean)
assert np.allclose(mean, manual_mean, atol=1e-7)


The posterior variance starts from prior variance 1 and subtracts covariance explained by the observations.

In [ ]:

K = kernel(X, X)
Ks = kernel(Xstar, X)
explained = Ks @ np.linalg.solve(K + 0.1 * np.eye(2), Ks.T)
manual_variance = 1.0 - float(explained[0, 0])
print("variance subtraction", manual_variance, var[0])
assert np.isclose(var[0], manual_variance, atol=1e-7)

near_short = math.exp(-0.5 * 1.0 ** 2 / 0.4 ** 2)
near_long = math.exp(-0.5 * 1.0 ** 2 / 2.0 ** 2)
assert near_long > near_short


## The dataset ladder

D1 is a tiny exact regression problem; D5 is deliberately ill-conditioned but still CPU-small.

In [ ]:

ladder = build_gp_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"])
    print("train", rung["X"].shape, "test", rung["Xstar"].shape)
    print("length", rung["length"], "noise", rung["sigma2"])
    print("first y", np.round(rung["y"][:3], 3))


In [ ]:

rows = []
means = []
vars_ = []
for i, rung in enumerate(ladder, start=1):
    kernel = lambda A, B, ell=rung["length"]: rbf_kernel(A, B, length_scale=ell, variance=1.0)
    mean, var, cov = gp_posterior(rung["X"], rung["y"], rung["Xstar"], kernel, sigma2=rung["sigma2"], jitter=1e-6)
    error = predictive_rmse(mean, rung["truth"])
    rows.append((f"D{i}", rung["name"], error))
    means.append(mean)
    vars_.append(var)

print("rung | predictive RMSE")
for label, name, error in rows:
    print(f"{label} | {name} | {error:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for i, rung in enumerate(ladder):
    ax = axes[0, i]
    x_axis = rung["Xstar"][:, 0]
    order = np.argsort(x_axis)
    mean = means[i][order]
    std = np.sqrt(vars_[i][order])
    truth = rung["truth"][order]
    ax.plot(x_axis[order], truth, label="truth")
    ax.plot(x_axis[order], mean, label="posterior")
    ax.fill_between(x_axis[order], mean - 2 * std, mean + 2 * std, alpha=0.2)
    ax.scatter(rung["X"][:, 0], rung["y"], s=16, color="black")
    ax.set_title(f"D{i + 1}")
    if i == 0:
        ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot([1, 2, 3, 4, 5], [row[2] for row in rows], marker="o")
ax.set_title("marginal-error / RMSE vs rung")
ax.set_xlabel("rung")
ax.set_ylabel("RMSE")
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: careless length-scale and conditioning

A long length-scale makes near-duplicate higher-dimensional points almost collinear. The fix is not to abandon the GP; sweep plausible length-scales and add jitter to stabilize the exact solve.

In [ ]:

d5 = ladder[-1]
lengths = [0.15, 0.4, 0.8, 1.5, 3.0]
errors = []
conds = []
for ell in lengths:
    kernel = lambda A, B, ell=ell: rbf_kernel(A, B, length_scale=ell, variance=1.0)
    K = kernel(d5["X"], d5["X"]) + d5["sigma2"] * np.eye(len(d5["X"]))
    conds.append(float(np.linalg.cond(K)))
    mean, var, cov = gp_posterior(d5["X"], d5["y"], d5["Xstar"], kernel, sigma2=d5["sigma2"], jitter=1e-5)
    errors.append(predictive_rmse(mean, d5["truth"]))

bad_error = errors[-1]
best_error = min(errors)
print("condition numbers", np.round(conds, 1))
print("bad long-scale error", bad_error)
print("best swept-scale error", best_error)
assert best_error <= bad_error


## Evaluate it + practice

- Metric: predictive mean RMSE or marginal NLL against the known synthetic truth.
- No-skill baseline: predict the training mean everywhere with constant variance.
- Cheap sanity check: posterior variance is no larger than prior variance at well-supported test points.
- Ablation: turn off jitter or fix the D5 length-scale at a careless value and error/conditioning worsens.
- Failure signals: negative variances, huge condition numbers, over-smoothed posterior, or confident extrapolation far from data.

**Practice.** Replace the RBF with a Matérn-like kernel and compare posterior bands.

**Practice.** Plot D5 error as a function of jitter size.

**Practice.** Add a duplicated training point and inspect the condition number.